In [2]:
import os
import numpy as np
from ase import Atoms
from ase.io import read
from dscribe.descriptors import SOAP

In [2]:
soap_descriptor = SOAP(
    species = ["Li", "P", "S"],
    periodic = True,
    sigma = 2.5,
    r_cut = 3.0,
    n_max = 4,
    l_max = 4,
    rbf = "polynomial",
    weighting = {'w0':0, 'function': 'pow', 'm': 3,'r0': 0.5, 'c' : 1, 'd':1},
    average="outer"
)

In [3]:
lattice_parameters = []
filename = 'dft_li3ps4.xyz'
with open(filename, 'r') as file:
    lines = file.readlines()
for line in lines:
    if line.startswith("Lattice"):
        parts = line.split()
        #get the first lattice parameter
        a = float(parts[0].split('"')[1])
        b = float(parts[4])
        c = float(parts[8].split('"')[0])
        pbc_boolean_1 = parts[-1].split('"')[0]
        pbc_boolean_2 = parts[-2]
        pbc_boolean_3 = parts[-3].split('"')[1]
        lattice_parameters.append([a, b, c, pbc_boolean_1, pbc_boolean_2, pbc_boolean_3])

In [4]:
train_soap_list = []
folder_path = "DFT_Li3PS4_Divided"
num_training_files = 16000
print("Processing...")
for i in range(1, num_training_files + 1):
    file_name = f"testing{i}.xyz"
    file_path = os.path.join(folder_path, file_name)
    if os.path.exists(file_path):
        if i % 100 == 0:
            print(f"Processing {file_name}...")
        new_molecule = read(file_path)
        a = lattice_parameters[i - 1][0]
        b = lattice_parameters[i - 1][1]
        c = lattice_parameters[i - 1][2]
        cell = np.array([[a, 0, 0], [0, b, 0], [0, 0, c]])
        new_molecule.set_cell(cell, scale_atoms=True)

        pbc_boolean_1 = lattice_parameters[i - 1][3]
        pbc_boolean_2 = lattice_parameters[i - 1][4]
        pbc_boolean_3 = lattice_parameters[i - 1][5]
        new_molecule.set_pbc((pbc_boolean_1, pbc_boolean_2, pbc_boolean_3))

        new_soap = soap_descriptor.create(new_molecule)
        train_soap_list.append(new_soap)

        # Save periodically to merge results into a single npy file
        if i % 100 == 0 or i == num_training_files:
            np.save("DFT_Li3PS4_outer_average_SOAP_16000.npy", np.array(train_soap_list))

    else:
        print(f"File {file_name} not found.")

# Final save in case there are remaining items
if len(train_soap_list) > 0:
    np.save("DFT_Li3PS4_outer_average_SOAP_16000.npy", np.array(train_soap_list))

Processing...
Processing testing100.xyz...
